# 03 — Fine-Tuning Experiments

Compares **zero-shot** vs **fine-tuned** performance for:
- TabPFN v2 (gradient-based fine-tuning)
- TabNet (self-supervised pretraining + supervised fine-tuning)
- FT-Transformer (full training from scratch with early stopping)

We use **German Credit** (1K rows) for the demo since it fits within TabPFN v2's 10K limit
and produces visible differences between zero-shot and fine-tuned performance.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data.loader import load_credit_dataset
from src.finetuning import FineTuningConfig, TabPFNFineTuner, TabNetFineTuner

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

DATASET = 'german_credit'
CHECKPOINT_DIR = Path('../results/.cache/checkpoints')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

X_train, X_val, X_test, y_train, y_val, y_test = load_credit_dataset(
    DATASET, return_val=True
)
print(f'Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}')

## 1. TabPFN v2: Zero-Shot vs Fine-Tuned

In [ ]:
try:
    from src.models.tabpfn_wrapper import TabPFNModel, FinetunedTabPFNModel
    
    # Zero-shot baseline
    model_zs = TabPFNModel(version='v2')
    result_zs = model_zs.evaluate(X_train, y_train, X_test, y_test, DATASET, 'zero_shot')
    print(f'TabPFN v2 zero-shot:  AUC={result_zs.auc_roc:.4f}  Time={result_zs.total_time:.1f}s')
    
    # Fine-tuned
    config = FineTuningConfig(
        epochs=30, lr=1e-5, batch_size=20,
        checkpoint_dir=CHECKPOINT_DIR,
        resume_from_checkpoint=True,
    )
    tuner = TabPFNFineTuner(version='v2')
    ft_result = tuner.fine_tune(None, X_train, y_train, config)
    
    ft_model = tuner.get_finetuned_model()
    from sklearn.metrics import roc_auc_score
    ft_proba = ft_model.predict_proba(X_test)
    if ft_proba.ndim > 1: ft_proba = ft_proba[:, 1]
    auc_ft = roc_auc_score(y_test, ft_proba)
    
    print(f'TabPFN v2 fine-tuned: AUC={auc_ft:.4f}  Val AUC={ft_result.final_val_auc:.4f}')
    print(f'Delta: {auc_ft - result_zs.auc_roc:+.4f}')

except ImportError as e:
    print(f'TabPFN not available: {e}')
    print('Install with: pip install tabpfn>=2.0')

## 2. TabNet: Supervised vs Pretrained+Finetuned

In [ ]:
try:
    from src.models.tabnet_wrapper import TabNetModel
    from sklearn.metrics import roc_auc_score
    
    # Supervised only
    model_sup = TabNetModel(pretrain=False)
    result_sup = model_sup.evaluate(X_train, y_train, X_test, y_test, DATASET, 'zero_shot')
    print(f'TabNet (supervised):  AUC={result_sup.auc_roc:.4f}  Time={result_sup.total_time:.1f}s')
    
    # Self-supervised pretraining + fine-tuning
    config = FineTuningConfig(
        epochs=100, batch_size=256, patience=20,
        checkpoint_dir=CHECKPOINT_DIR,
        resume_from_checkpoint=True,
    )
    tuner = TabNetFineTuner()
    ft_result = tuner.fine_tune(None, X_train, y_train, config)
    
    ft_model = tuner.get_finetuned_model()
    from src.finetuning.tabnet_finetune import TabNetFineTuner as TNFt
    X_test_np = TNFt._prepare(X_test)
    ft_proba = ft_model.predict_proba(X_test_np)[:, 1]
    auc_ft = roc_auc_score(y_test, ft_proba)
    
    print(f'TabNet (pretrain+ft): AUC={auc_ft:.4f}  Val AUC={ft_result.final_val_auc:.4f}')
    print(f'Delta: {auc_ft - result_sup.auc_roc:+.4f}')

except ImportError as e:
    print(f'TabNet not available: {e}')
    print('Install with: pip install pytorch-tabnet torch>=2.1')

## 3. Summary: Fine-Tuning Impact Across Models

In [ ]:
# Collect results into a comparison table
# (Fill in from cells above)
comparison_data = {
    'Model': ['TabPFN-v2', 'TabPFN-v2', 'TabNet', 'TabNet'],
    'Phase': ['Zero-Shot', 'Fine-Tuned', 'Supervised', 'Pretrain+FT'],
    'AUC-ROC': [None, None, None, None],  # Replace with actual values from above
}

df_cmp = pd.DataFrame(comparison_data)
print('Fine-tuning impact on German Credit:')
print(df_cmp.to_string(index=False))

## 4. Loss Curve Visualisation

In [ ]:
# Requires ft_result from Section 2 (TabNet) to have train_loss_curve populated
try:
    if hasattr(ft_result, 'train_loss_curve') and ft_result.train_loss_curve:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        
        axes[0].plot(ft_result.train_loss_curve, label='Train Loss')
        if ft_result.val_loss_curve:
            axes[0].plot(ft_result.val_loss_curve, label='Val Loss')
        if ft_result.best_epoch:
            axes[0].axvline(ft_result.best_epoch - 1, color='red', linestyle='--',
                            label=f'Best epoch ({ft_result.best_epoch})')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Loss')
        axes[0].set_title('Training Loss Curve')
        axes[0].legend()
        
        # Fine-tuning summary
        result_dict = ft_result.to_dict()
        info_text = '\n'.join([f'{k}: {v}' for k, v in result_dict.items() if v is not None])
        axes[1].text(0.1, 0.5, info_text, transform=axes[1].transAxes,
                    fontfamily='monospace', fontsize=10,
                    verticalalignment='center')
        axes[1].set_title('Fine-Tuning Result Summary')
        axes[1].axis('off')
        
        plt.tight_layout()
        plt.savefig('../results/figures/03_loss_curve.png', bbox_inches='tight')
        plt.show()
    else:
        print('No loss curve available (TabPFN does not expose epoch-level curves).')
except NameError:
    print('Run cells 1 and 2 first to generate ft_result.')